In [1]:
setwd("/well/lindgren-ukbb/projects/ukbb-11867/flassen/projects/KO/wes_ko_ukbb")
devtools::load_all('utils/modules/R/prstools')
library(argparse)

i Loading PRStools

Loading required package: bigsnpr

Loading required package: bigstatsr

Loading required package: data.table

Loading required package: bigassertr



In [2]:
args <- list(
    chrom = "chr21",
    pred = "data/prs/hapmap/ukb_500k/validation/ukb_hapmap_500k_eur_chr21.bed",
    ldsc = "data/prs/ldsc/ldsc_Alanine_aminotransferase_residual_int.rds",
    ld_dir = "data/prs/hapmap/ld/matrix_unrel_kin",
    method = 'auto',
    out_prefix = "data/prs/scores/auto/prs_auto_Alanine_aminotransferase_residual_int",
    tmp_bfile = "data/prs/scores/auto/prs_auto_Alanine_aminotransferase_residual_int_chr21.bfile",
    standardized_gt = TRUE
)

In [3]:
stopifnot(file.exists(args$pred))
stopifnot(file.exists(args$ldsc))
stopifnot(!file.exists(args$tmp_bfile))
stopifnot(dir.exists(args$ld_dir))
stopifnot(args$chrom %in% paste0("chr",1:22))
stopifnot(dir.exists(dirname(args$out_prefix)))
stopifnot(args$method %in% c('inf', 'auto'))

log <- paste0(args$out_prefix,".log")

# setup parallel environment
NCORES <- max(1, nb_cores())
bigparallelr::assert_cores(NCORES)

# laod ldsc and qced GWAS
ldsc <- readRDS(args$ldsc)
gwas <- ldsc$gwas
h2 <- ldsc$coefficients$estimate[2]
pvalue <- ldsc$coefficients$pvalue[2]

# Estimate h2 chromosome-wide
N_total <- nrow(gwas)
N_chr <- sum(gwas$chr == args$chrom)
h2_init <- h2 * (N_chr / N_total)

# check estimates and ensure that
# the trait is actually heritable
stopifnot(pvalue < 1e-5)
stopifnot(!is.null(gwas))
print(nrow(gwas))

[1] 1165296


# *** new strategy ***
We will need to somehow filter the snp (LD-matrix) alongside the GWAS file and ENSURE that gwas positions are still matched between the two. Can we avoid this without creating another file backed LD-matrix? We could re-create the LD reference and subset to variants in our validation set, but what if we need to run on a new set? Need to think more about this.

In [4]:
# get pred file
pred <- load_bigsnp_from_bed(args$pred) # A data.frame: 16713 × 5

In [7]:
head(pred$map$rsid)
head(gwas$rsid)

[1] "chr21:10510446:A:G" "chr21:13222943:T:A" "chr21:13258267:A:G"
[4] "chr21:13268079:G:A" "chr21:13270143:A:G" "chr21:13277477:C:A"

[1] "chr1:817341:A:G" "chr1:818802:A:G" "chr1:818954:T:C" "chr1:825532:C:T"
[5] "chr1:833068:G:A" "chr1:841166:A:G"

In [8]:
gwas <- gwas[gwas$rsid %in% pred$map$rsid,]

In [66]:
# get SNP correlations and LD
snp <- get_single_ld_matrix(gwas, chr = args$chrom, ld_dir = args$ld_dir)

# match GWAS with snp-correlation map
gwas <- gwas[na.omit(match(snp$map$marker, gwas$marker)), ]

# check that LD-matrix markers and gwas markers have overlap
# Check that ordering of markers are actually matching
stopifnot(all(gwas$marker %in% snp$map$marker))
stopifnot(all(snp$map$marker %in% gwas$marker))
stopifnot(sum(gwas$marker == snp$map$marker) / nrow(gwas) == 1)

# match prediction data 
bfile <- tempfile(tmpdir = dirname(args$out_prefix))
pred <- match_bigsnp_with_gwas(obj=pred, gwas=gwas, bfile=args$tmp_bfile)




In [79]:
stopifnot(pred$map$rsid == rev(gwas$rsid))

ERROR: Error: pred$map$rsid == rev(gwas$rsid) are not all TRUE


In [71]:
pred$map$rsid <- gsub("_",":", pred$map$rsid.ss)

In [47]:
genotypes <- pred$genotypes
indicies <- pred$gwas_indicies

In [ ]:
# check that we have genotypes
stopifnot(!is.null(genotypes))
stopifnot(!is.null(indicies))

# dimensions
cols <- genotypes$`.->ncol` # variants
rows <- genotypes$`.->nrow` # samples
missing_gt <- NA

In [6]:
obj <- pred
gwas <- gwas
ind_row = NULL
bfile = NULL
ncores = 1

# add indexes used for later subsetting
bed_map <- obj$map
bed_map$marker <- get_ldpred_marker(bed_map)
bed_map$index <- 1:nrow(bed_map)
gwas$marker <- get_ldpred_marker(gwas)
gwas$index <- 1:nrow(gwas)

# check that all markers are contained in one another
stopifnot(sum(gwas$marker %in% bed_map$marker) == sum(bed_map$marker %in% gwas$marker))

# remove any bed markers not in gwas (assuming
# that the GWAS has already been qced before)
bed_map <- bed_map[bed_map$marker %in% gwas$marker,]

# match gwas betas with bed map
indicies <- match(bed_map$marker, gwas$marker)
gwas_matched <- gwas[indicies,]
gwas_matched <- gwas_matched[!(rowSums(is.na(gwas_matched)) == ncol(gwas_matched)),]

# check that all positions match
stopifnot(all(gwas_matched$pos == bed_map$pos))

# subset bigsnp to get the right genotypes
samples <- obj$bigsnp$fam$sample.ID
if (is.null(ind_row)) ind_row <- 1:length(samples)
subsetted_bigsnp <- snp_subset(obj$bigsnp,
                               ind.col = bed_map$index,
                               ind.row = ind_row,
                               backingfile = bfile
                               )
subsetted_bigsnp <- snp_attach(subsetted_bigsnp)
genotypes <- subsetted_bigsnp$genotypes
stopifnot(dim(genotypes)[2] == nrow(gwas_matched))


Warning message in seq_len(nrow(x)):
"first element used of 'length.out' argument"


ERROR: Error in seq_len(nrow(x)): argument must be coercible to non-negative integer


In [ ]:
cols

In [ ]:
nrow(gwas)

In [16]:
# need to impute missing SNPs
if (!is.null(args$impute)){
    sum_cols <- lapply(1:cols, function(i) return(sum(is.na(genotypes[,i]))))
    missing_gt <- sum(unlist(sum_cols))
    genotypes <- snp_fastImputeSimple(genotypes, method = args$impute)
}



In [19]:
# standardize genotypes
means <- NULL; sds <- NULL
if (args$standardized_gt){
 means <- as.numeric(unlist(lapply(1:cols, function(i) mean(genotypes[,i], na.rm = TRUE))))
 sds <- as.numeric(unlist(lapply(1:cols, function(i) sd(genotypes[,i], na.rm = TRUE))))
}

In [20]:
write("Running multi_auto", stderr())
    multi_auto <- snp_ldpred2_auto(
    corr = snp$corr,
    df_beta = gwas,
    h2_init = h2_init,
    vec_p_init = seq_log(1e-4, 0.9, NCORES), # using cores instead 30
    ncores = NCORES)

# save data chains
multi_auto_path <- paste0(args$out_prefix,'_chains.rda')
saveRDS(multi_auto, multi_auto_path, compress = 'xz')

In [ ]:
# plot chains
 auto <- multi_auto[[1]]
 p <- plot_grid(
    qplot(y = auto$path_p_est) +
      theme_bigstatsr() +
      geom_hline(yintercept = auto$p_est, col = "blue") +
      scale_y_log10() +
      labs(y = "p"),
    qplot(y = auto$path_h2_est) +
      theme_bigstatsr() +
      geom_hline(yintercept = auto$h2_est, col = "blue") +
      labs(y = "h2"),
    ncol = 1, align = "hv"
  )
 # save plot
 outplot <-
 ggsave(plot=p,filename=paste0(args$outfile, "_chains.png"))

In [21]:
# get estimates with indicies corresponding to pred genotypes
beta_auto <- sapply(multi_auto, function(auto){
  auto$beta_est})

# perform matrix multiplication
pred_auto <- big_prodMat(
    genotypes,
    beta_auto,
    center = means,
    scale = sds,
    ncores = NCORES)

# quality controls on chains
sc <- apply(pred_auto, 2, sd, na.rm = TRUE)
keep <- abs(sc - median(sc)) < 3 * mad(sc)
final_beta_auto <- rowMeans(beta_auto[, keep], na.rm = TRUE)

ERROR: Error: Incompatibility between dimensions.
'ind.col' and 'rows_along(A.col)' should have the same length.


In [26]:
cols

[1] 16673

In [24]:
dim(beta_auto)

[1] 16688     1

In [ ]:
# get final predicton
final_pred_auto <- big_prodVec(
    genotypes,
    final_beta_auto,
    center = means,
    scale = sds,
    ncores = NCORES)

    


In [ ]:
# keep
final_pred <- final_pred_auto
beta_out <- cbind(gwas, final_beta_auto)
fwrite(beta_out, file = paste0(args$out_prefix,"_betas.txt.gz"), sep = '\t')

In [6]:
# check that LD-matrix markers and gwas markers have overlap
# Check that ordering of markers are actually matching
stopifnot(all(gwas$marker %in% snp$map$marker))
stopifnot(all(snp$map$marker %in% gwas$marker))
stopifnot(sum(gwas$marker == snp$map$marker) / nrow(gwas) == 1)

# load data to be used for prediction
bfile <- tempfile(tmpdir = dirname(args$out_prefix))
pred <- load_bigsnp_from_bed(args$pred)
pred <- match_bigsnp_with_gwas(obj=pred, gwas=gwas, bfile=args$tmp_bfile)
genotypes <- pred$genotypes
indicies <- pred$gwas_indicies

stopifnot(!is.null(genotypes))
stopifnot(!is.null(indicies))

In [7]:
# count variants with missing GTs
cols <- genotypes$`.->ncol`
rows <- genotypes$`.->nrow`
sum_rows <- lapply(1:rows, function(i)
    return(sum(is.na(genotypes[i,])))
)

missing_gt <- sum(unlist(sum_rows))
pct <- paste0(round(missing_gt/rows, 4)*100)


In [14]:
pct <- round(missing_gt/rows, 4)*100

In [15]:
pct

[1] 2.75

In [18]:
write("Running multi_auto", stderr())
multi_auto <- snp_ldpred2_auto(
    corr = snp$corr,
    df_beta = gwas,
    h2_init = h2_init,
    vec_p_init = seq_log(1e-4, 0.9, 30),
    ncores = NCORES)

# save data chains
#multi_auto_path <- paste0(args$out_prefix,'_chains.rda')
#multi_auto <- readRDS(multi_auto_path)

In [19]:
str(multi_auto)

List of 30
 $ :List of 9
  ..$ beta_est   : num [1:16307] -4.76e-05 -6.45e-06 -6.51e-06 -5.77e-06 -1.52e-06 ...
  ..$ postp_est  : num [1:16307] 0.214 0.212 0.212 0.212 0.211 ...
  ..$ sample_beta: num[1:16307, 0 ] 
  ..$ p_est      : num 0.214
  ..$ h2_est     : num 0.00123
  ..$ path_p_est : num [1:1500] 7.25e-05 1.63e-04 3.75e-04 1.42e-04 2.95e-04 ...
  ..$ path_h2_est: num [1:1500] 0.0001 0.0001 0.00019 0.000104 0.000199 ...
  ..$ h2_init    : num 0.00592
  ..$ p_init     : num 1e-04
 $ :List of 9
  ..$ beta_est   : num [1:16307] -4.60e-05 -6.14e-06 -6.27e-06 -5.53e-06 -1.41e-06 ...
  ..$ postp_est  : num [1:16307] 0.152 0.149 0.149 0.149 0.148 ...
  ..$ sample_beta: num[1:16307, 0 ] 
  ..$ p_est      : num 0.151
  ..$ h2_est     : num 0.0012
  ..$ path_p_est : num [1:1500] 5.98e-05 1.49e-05 1.17e-05 4.94e-05 2.85e-04 ...
  ..$ path_h2_est: num [1:1500] 1e-04 1e-04 1e-04 1e-04 1e-04 ...
  ..$ h2_init    : num 0.00592
  ..$ p_init     : num 0.000137
 $ :List of 9
  ..$ beta_est   : 

In [20]:
test <- snp_fastImputeSimple(genotypes)

ERROR: Error: You don't have write permissions for this FBM.


In [ ]:
# get estimates with indicies corresponding to pred genotypes
beta_auto <- sapply(multi_auto, function(auto){
      auto$beta_est})

# perform matrix multiplication
pred_auto <- big_prodMat(
    test,
    beta_auto,
    ncores = NCORES)

In [53]:
# quality controls on chains
na_rows <- rowSums(is.na(pred_auto)) > 0 
na_rows_pct <- round(100*(sum(na_rows) / nrow(pred_auto)), 2)
write(paste0(na_rows_pct,"% of chain rows contains NAs (", args$out_prefix, ")" ), stdout())





0% of chain rows contains NAs (data/prs/scores/auto/prs_auto_BMI_chr21)


In [55]:
sum(M)

[1] 10488

In [ ]:
sc <- apply(pred_auto, 2, sd, na.rm = TRUE)
keep <- abs(sc - median(sc)) < 3 * mad(sc)
final_beta_auto <- rowMeans(beta_auto[, keep], na.rm = TRUE)

In [ ]:
# get final predicton
final_pred_auto <- big_prodVec(
   genotypes,
   final_beta_auto,
   ncores = NCORES)

final_pred <- final_pred_auto